In [1]:
import os

In [2]:
%pwd

'd:\\Data_Science\\Projects\\Chicken-Disease-Classification-\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\Data_Science\\Projects\\Chicken-Disease-Classification-'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list

In [7]:
@dataclass
class PrepareCallBacksConfig:
    root_dir: Path
    tenserboard_root_log_dir: Path
    checkpoint_model_filepath: Path

In [10]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import create_directories,read_yaml,save_json
import tensorflow as tf

In [31]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

    def get_prepare_callbacks_config(self) -> PrepareCallBacksConfig:
        prepare_callbacks_config = self.config.prepare_callbacks
        model_ckpt_dir = os.path.dirname(prepare_callbacks_config.checkpoint_model_filepath)
        create_directories([Path(model_ckpt_dir), Path(prepare_callbacks_config.tenserboard_root_log_dir)])
        return PrepareCallBacksConfig(
            root_dir=Path(prepare_callbacks_config.root_dir),
            tenserboard_root_log_dir=Path(prepare_callbacks_config.tenserboard_root_log_dir),
            checkpoint_model_filepath=Path(prepare_callbacks_config.checkpoint_model_filepath)
        )
    
    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir,'Chicken-fecal-images')
        create_directories([training.root_dir])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.update_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config

In [12]:
import time

In [13]:
class PrepareCallBacks:
    def __init__(self, config: PrepareCallBacksConfig):
        self.config = config

    @property
    def _create_tb_callbacks(self):
        timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
        tb_logs_dir = os.path.join(self.config.tenserboard_root_log_dir, f"tb_logs_at{timestamp}")
        tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=tb_logs_dir)
        return tensorboard_callback
    
    @property
    def _create_checkpoint_callbacks(self):
        checkpoint_filepath = self.config.checkpoint_model_filepath
        model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_filepath, save_best_only=True)
        return model_checkpoint_callback
    
    def get_tb_ckpt_callbacks(self):
        return [self._create_tb_callbacks, self._create_checkpoint_callbacks]

In [29]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(self.config.updated_base_model_path)

    def train_valid_generator(self):
        datagenerator_kwargs = dict(
            rescale=1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size = self.config.params_image_size[:-1],
            batch_size = self.config.params_batch_size,
            interpolation = 'bilinear'
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(**datagenerator_kwargs)
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset='validation',
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                horizontal_flip=True,
                rotation_range=40,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )

        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset='training',
            shuffle=True,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)
    
    def train(self, callbacks_list: list):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size
        
        self.model.fit(
            self.train_generator,
            steps_per_epoch=self.steps_per_epoch,
            validation_data=self.valid_generator,
            validation_steps=self.validation_steps,
            epochs=self.config.params_epochs,
            callbacks=callbacks_list
        )
        self.save_model(path=self.config.trained_model_path, model=self.model)


In [33]:
try:
    config = ConfigurationManager()
    prepare_callbacks_config = config.get_prepare_callbacks_config()
    prepare_callbacks = PrepareCallBacks(config=prepare_callbacks_config)
    callbacks_list = prepare_callbacks.get_tb_ckpt_callbacks()

    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train(callbacks_list=callbacks_list)

except Exception as e:
    raise e

[2026-02-17 11:51:18,724 - INFO - common - yaml file: config\config.yaml loaded successfully]
[2026-02-17 11:51:18,726 - INFO - common - yaml file: params.yaml loaded successfully]
[2026-02-17 11:51:18,727 - INFO - common - created directory at: artifacts\prepare_callbacks\checkpoint_dir]
[2026-02-17 11:51:18,730 - INFO - common - created directory at: artifacts\prepare_callbacks\tensorboard_log_dir]
[2026-02-17 11:51:18,732 - INFO - common - created directory at: artifacts/training]
Found 78 images belonging to 2 classes.
Found 312 images belonging to 2 classes.
19/19 ━━━━━━━━━━━━━━━━━━━━ 43s 2s/step - accuracy: 0.7697 - loss: 3.9767 - val_accuracy: 0.8125 - val_loss: 2.6501
